# SEBI Instruction Dataset Generation

Runs entirely in Colab on a free T4 GPU. ~2-3 hours for 14K prompts.

**Setup**: Runtime → Change runtime type → **T4 GPU**

In [ ]:
# Cell 1: Check GPU
!nvidia-smi
import torch
print(f"\nGPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Upload sebi_colab_data.zip
# Option A: Use the file browser (folder icon on left → Upload)
# Option B: Mount Google Drive (uncomment below)

# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/sebi_colab_data.zip /content/

import zipfile, os
zip_path = "/content/sebi_colab_data.zip"
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    n = len([f for f in os.listdir('/content/SEBI_Canonical/') if f.endswith('.json')])
    print(f"Extracted! {n} canonical docs found.")
else:
    print("Upload sebi_colab_data.zip first!")

In [ ]:
# Cell 3: Write the generation script
script = r'''"""SEBI Instruction Dataset Generation - Colab T4"""
import json, re, os, sys, time, hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DATA_ROOT = "/content/SEBI_Canonical"
CHECKPOINT_PATH = "/content/output/raw_checkpoint.jsonl"
FINAL_PATH = "/content/output/sebi_instructions.jsonl"
MIN_SECTION_CHARS = 500
MAX_SECTION_CHARS = 6000
MAX_CHARS_PER_BATCH = 10000
MAX_SECTIONS_PER_BATCH = 5
BATCH_SIZE = 16
MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.4
TOP_P = 0.9
MAX_MODEL_LEN = 4096

@dataclass
class Section:
    section_id: str
    section_type: str
    number: str
    title: str
    text: str
    parent_id: Optional[str] = None
    parent_type: Optional[str] = None
    parent_number: Optional[str] = None
    depth: int = 0

STRUCTURAL_TAGS = {"chapter","regulation","section","rule","clause","schedule","annexure","part","form","preamble","order_header","recital","direction","penalty","text"}
DEPTH_MAP = {"preamble":0,"order_header":0,"chapter":1,"part":1,"regulation":2,"section":2,"rule":2,"schedule":2,"annexure":2,"form":3,"clause":3,"recital":1,"direction":1,"penalty":1,"text":0}

def _strip_xml_tags(text):
    return re.sub(r'<[^>]+>', '', text).strip()

def _unescape_xml(text):
    return text.replace('&amp;','&').replace('&lt;','<').replace('&gt;','>').replace('&quot;','"').replace('&apos;',"'")

def _extract_number_from_text(text, section_type):
    m = re.match(r'^\s*(\d+[A-Z]?)\s*\.\s', text)
    if m:
        return m.group(1)
    m = re.match(rf'^\s*{section_type.capitalize()}\s+(\d+[A-Z]?)\b', text, re.IGNORECASE)
    return m.group(1) if m else ""

def parse_structured_text(structured_text, document_id):
    sections = []
    current_chapter = None
    idx = 0
    body_match = re.search(r'<body>(.*?)</body>', structured_text, re.DOTALL)
    body = body_match.group(1) if body_match else structured_text
    tag_names = "|".join(STRUCTURAL_TAGS)
    open_pattern = re.compile(rf'<(?P<tag>{tag_names})(?P<attrs>[^>]*)>', re.IGNORECASE)
    matches = list(open_pattern.finditer(body))
    if not matches:
        text = _strip_xml_tags(body)
        if text.strip():
            sections.append(Section(f"{document_id}/sec_0","text","","",text,depth=0))
        return sections
    for m in matches:
        tag = m.group("tag").lower()
        attrs = m.group("attrs")
        num_match = re.search(r'number="([^"]*)"', attrs)
        title_match = re.search(r'title="([^"]*)"', attrs)
        number = num_match.group(1) if num_match else ""
        title = _unescape_xml(title_match.group(1)) if title_match else ""
        content_start = m.end()
        close_pos = body.find(f"</{m.group('tag')}>", content_start)
        content_end = close_pos if close_pos != -1 else len(body)
        text = _unescape_xml(_strip_xml_tags(body[content_start:content_end]))
        if not text.strip():
            continue
        if not number and tag in ("regulation","section","rule"):
            number = _extract_number_from_text(text, tag)
        if title and text.strip().startswith(title.strip()[:50]):
            title = ""
        sid = f"{document_id}/sec_{idx}"
        idx += 1
        pid = ptype = pnum = None
        if tag in ("regulation","section","rule","clause") and current_chapter:
            pid = current_chapter.section_id
            ptype = current_chapter.section_type
            pnum = current_chapter.number
        sec = Section(sid, tag, number, title, text, pid, ptype, pnum, DEPTH_MAP.get(tag, 0))
        sections.append(sec)
        if tag == "chapter":
            current_chapter = sec
    return sections

def split_document(doc):
    did = doc.get("document_id", "unknown")
    st = doc.get("structured_text", "")
    if not st:
        t = doc.get("text", "")
        if t.strip():
            return [Section(f"{did}/sec_0", "text", "", doc.get("title", ""), t, depth=0)]
        return []
    secs = parse_structured_text(st, did)
    filtered = [s for s in secs if len(s.text) >= 50]
    return filtered if filtered else secs

TASK_GROUPS = [
    ["definition", "plain_english", "summary", "applicability"],
    ["key_obligations", "compliance_checklist", "entity_understanding", "legal_citation"],
    ["scenario", "violation_detection", "cross_reference", "reasoning_chain"],
]

TASK_DESCS = {
    "definition": 'definition - Define key terms. Output: {"term","definition","citation"}',
    "plain_english": "plain_english - Explain in plain English. Output: text",
    "summary": 'summary - Four levels. Output: {"one_sentence","three_sentence","five_bullets":[],"detailed"}',
    "key_obligations": 'key_obligations - Extract obligations. Output: {"obligations":[{"entity","obligation","deadline","reporting_requirement"}]}',
    "compliance_checklist": 'compliance_checklist - Checklist. Output: {"checklist":[{"category","item","citation"}]}',
    "applicability": 'applicability - Who does this apply to? Output: {"applies_to":[],"exemptions":[],"effective_when","conditions":[],"not_applicable_when"}',
    "scenario": 'scenario - 3 scenarios (easy/medium/hard). Output: {"scenario","question","applicable_regulation","reasoning","conclusion","difficulty"} per scenario',
    "violation_detection": 'violation_detection - 3 scenarios. Output: {"violation_status","scenario","reasoning","remediation"} per scenario',
    "cross_reference": 'cross_reference - References. Output: {"cross_references":[{"referenced_document","reference_type","relationship","explanation"}]}',
    "entity_understanding": 'entity_understanding - Entity roles. Output: {"entities":[{"entity_name","entity_type","role_in_section","obligations":[]}]}',
    "legal_citation": "legal_citation - Answer with citations. Output: text",
    "reasoning_chain": 'reasoning_chain - Facts->Regs->Analysis->Conclusion. Output: {"facts","applicable_regulations":[],"analysis","conclusion","references":[]}',
}

SYS_PROMPT = """You are a SEBI regulatory expert. Generate instruction-following examples from legal document sections.
Rules: Support claims with source text. Cite regulation/section numbers. Output valid JSON only.
Tasks (Group {group_index} of {total_groups}):
{task_descriptions}"""

USR_PROMPT = """Document: {document_title} ({document_type}, {domain})

## Sections
{sections_json}

## Output Format
JSON array, each element:
{{"section_index": 0, "task_type": "...", "difficulty": "easy|medium|hard|standard", "instruction": "...", "input": "", "output": "...", "citations": []}}

Generate JSON array now."""

def render(t, v):
    return re.sub(r'\{\{\s*(\w+)\s*\}\}', lambda m: str(v.get(m.group(1).strip(), "")), t)

def build_sections_json(sections):
    return json.dumps([{"index": i, "section_type": s.section_type, "number": s.number, "title": s.title, "text": s.text} for i, s in enumerate(sections)], ensure_ascii=False)

def create_batches(sections):
    batches = []
    cur = []
    chars = 0
    for s in sections:
        t = s.text
        if len(t) > MAX_SECTION_CHARS:
            t = t[:MAX_SECTION_CHARS] + "\n[...]"
        if cur and (chars + len(t) > MAX_CHARS_PER_BATCH or len(cur) >= MAX_SECTIONS_PER_BATCH):
            batches.append(cur)
            cur = []
            chars = 0
        cur.append(Section(s.section_id, s.section_type, s.number, s.title, t, s.parent_id, s.parent_type, s.parent_number, s.depth))
        chars += len(t)
    if cur:
        batches.append(cur)
    return batches

REASONING_KW = ["because","therefore","since","thus","hence","accordingly","due to","based on","under","pursuant to","as per","requires","shall","must","obligated","prohibited","compliance","regulation","section","rule","act","circular"]

def parse_output(output):
    if not output or not output.strip():
        return None
    try:
        p = json.loads(output)
        return p if isinstance(p, list) else [p]
    except:
        pass
    m = re.search(r'```(?:json)?\s*\n?(.*?)\n?```', output, re.DOTALL)
    if m:
        try:
            p = json.loads(m.group(1))
            return p if isinstance(p, list) else [p]
        except:
            pass
    s = output.find('[')
    e = output.rfind(']')
    if s != -1 and e > s:
        try:
            return json.loads(output[s:e+1])
        except:
            pass
    return None

def validate(ex_raw, doc, sections, model_id, seen):
    if not isinstance(ex_raw, dict):
        return None, "not_dict"
    tt = ex_raw.get("task_type", "")
    if not tt:
        return None, "no_task_type"
    si = ex_raw.get("section_index", 0)
    try:
        si = int(si) if isinstance(si, str) else si
    except:
        si = 0
    if si < 0 or si >= len(sections):
        si = 0
    sec = sections[si]
    inst = ex_raw.get("instruction", "")
    out = ex_raw.get("output", "")
    if isinstance(out, (dict, list)):
        out = json.dumps(out, ensure_ascii=False)
    else:
        out = str(out)
    if len(out) < 100:
        return None, "too_short"
    if not inst.strip():
        return None, "empty_instruction"
    rs = sum(1 for kw in REASONING_KW if kw in out.lower())
    if rs < 2:
        return None, "no_reasoning"
    dk = hashlib.sha256((inst[:200] + out[:200]).encode()).hexdigest()[:16]
    if dk in seen:
        return None, "duplicate"
    seen.add(dk)
    cits = ex_raw.get("citations", [])
    if not cits:
        cits = sorted(set(m.group(0) for pat in [r'(?:Regulation|Reg\.)\s+\d+[A-Z]?', r'(?:Section|Sec\.)\s+\d+[A-Z]?', r'(?:Rule)\s+\d+[A-Z]?', r'Chapter\s+[IVXLCDM]+', r'Schedule\s+[IVXLCDM]+'] for m in re.finditer(pat, out, re.IGNORECASE)))
    eid = hashlib.sha256(f"{doc['document_id']}|{sec.section_id}|{tt}|{inst[:200]}".encode()).hexdigest()[:16]
    return {"id": eid, "document_id": doc.get("document_id", ""), "document_type": doc.get("document_type", ""), "domain": doc.get("domain", ""), "task_type": tt, "difficulty": ex_raw.get("difficulty", "standard"), "source_section": f"{sec.section_type} {sec.number}".strip(), "source_section_id": sec.section_id, "instruction": inst, "input": ex_raw.get("input", ""), "output": out, "citations": cits if isinstance(cits, list) else [], "references": [], "entities": [], "generated_by": "colab_transformers", "model_version": model_id, "prompt_version": "1.0.0", "quality_score": min(1.0, rs / 10.0)}, None

def main():
    print("=" * 60)
    print(f"SEBI Dataset Generation | Model: {MODEL_ID} | Batch: {BATCH_SIZE}")
    print("=" * 60)

    root = Path(DATA_ROOT)
    docs = []
    for p in sorted(root.rglob("*.json")):
        if p.name.startswith("_") or p.name == "pipeline_report.json":
            continue
        try:
            d = json.loads(p.read_text())
        except:
            continue
        if d.get("status") == "ok":
            docs.append(d)
    print(f"Loaded {len(docs)} docs")

    all_prompts = []
    for doc in docs:
        secs = [s for s in split_document(doc) if len(s.text) >= MIN_SECTION_CHARS]
        if not secs:
            continue
        for batch in create_batches(secs):
            sj = build_sections_json(batch)
            for gi, tg in enumerate(TASK_GROUPS):
                td = "\n".join(f"{i+1}. {TASK_DESCS.get(t, t)}" for i, t in enumerate(tg))
                v = {"document_title": doc.get("title", ""), "document_type": doc.get("document_type", ""), "domain": doc.get("domain", ""), "group_index": str(gi + 1), "total_groups": str(len(TASK_GROUPS)), "task_descriptions": td, "sections_json": sj}
                all_prompts.append(([{"role": "system", "content": render(SYS_PROMPT, v)}, {"role": "user", "content": render(USR_PROMPT, v)}], doc, batch))
    print(f"Built {len(all_prompts)} prompts")

    Path(CHECKPOINT_PATH).parent.mkdir(parents=True, exist_ok=True)
    raw_outputs = []
    if Path(CHECKPOINT_PATH).exists():
        with open(CHECKPOINT_PATH) as f:
            for line in f:
                if line.strip():
                    raw_outputs.append(json.loads(line)["output"])
        print(f"Resumed: {len(raw_outputs)} already done")

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    print(f"\nLoading {MODEL_ID}...")
    tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
    model.eval()
    print(f"Loaded on {model.device}")

    remaining = all_prompts[len(raw_outputs):]
    print(f"Generating {len(remaining)} prompts (batch={BATCH_SIZE}, max_tokens={MAX_NEW_TOKENS})...")
    t0 = time.time()
    total_b = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

    with open(CHECKPOINT_PATH, "a") as rf:
        for bi in range(total_b):
            s = bi * BATCH_SIZE
            e = min(s + BATCH_SIZE, len(remaining))
            pts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m, _, _ in remaining[s:e]]
            inp = tok(pts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_MODEL_LEN).to(model.device)
            with torch.no_grad():
                outs = model.generate(**inp, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_p=TOP_P, do_sample=True, repetition_penalty=1.05, pad_token_id=tok.pad_token_id)
            il = inp["input_ids"].shape[1]
            for i, o in enumerate(outs):
                txt = tok.decode(o[il:], skip_special_tokens=True)
                raw_outputs.append(txt)
                rf.write(json.dumps({"index": len(raw_outputs) - 1, "output": txt}, ensure_ascii=False) + "\n")
                rf.flush()
            el = time.time() - t0
            done = len(raw_outputs)
            rate = e / el if el > 0 else 0
            eta = (len(all_prompts) - done) / rate / 60 if rate > 0 else 0
            print(f"  {bi+1}/{total_b} | {done}/{len(all_prompts)} ({done/len(all_prompts)*100:.1f}%) | {rate:.1f}/s | ETA: {eta:.0f}m", flush=True)

    print(f"\nGeneration done: {len(raw_outputs)} in {(time.time()-t0)/60:.1f}m")

    print("\nParsing & validating...")
    seen = set()
    examples = []
    Path(FINAL_PATH).parent.mkdir(parents=True, exist_ok=True)
    stats = {"valid": 0, "rejected": 0, "reasons": {}}
    with open(FINAL_PATH, "w") as f:
        for i, (raw, (msgs, doc, batch)) in enumerate(zip(raw_outputs, all_prompts)):
            exs = parse_output(raw)
            if exs is None:
                stats["rejected"] += 1
                stats["reasons"]["invalid_json"] = stats["reasons"].get("invalid_json", 0) + 1
                continue
            for er in exs:
                ex, reason = validate(er, doc, batch, MODEL_ID, seen)
                if ex:
                    examples.append(ex)
                    f.write(json.dumps(ex, ensure_ascii=False) + "\n")
                    stats["valid"] += 1
                else:
                    stats["rejected"] += 1
                    stats["reasons"][reason] = stats["reasons"].get(reason, 0) + 1
            if (i + 1) % 500 == 0:
                print(f"  {i+1}/{len(raw_outputs)} | Valid: {stats['valid']} | Rejected: {stats['rejected']}")

    print(f"\n{'='*60}")
    print(f"COMPLETE: {stats['valid']} valid | {stats['rejected']} rejected")
    if stats["reasons"]:
        for r, c in sorted(stats["reasons"].items(), key=lambda x: -x[1]):
            print(f"  {r}: {c}")
    print(f"Saved: {FINAL_PATH}")
    print(f"{'='*60}")

if __name__ == "__main__":
    main()
'''

with open('/content/colab_vllm_generate.py', 'w') as f:
    f.write(script)
print("Script written!")

In [ ]:
# Cell 4: Run generation (~2-3 hours)
# Saves checkpoint after every batch — if Colab disconnects, just re-run this cell
%cd /content
!python colab_vllm_generate.py

In [ ]:
# Cell 5: Check results
import json
from collections import Counter

path = "/content/output/sebi_instructions.jsonl"
examples = []
with open(path) as f:
    for line in f:
        examples.append(json.loads(line))

print(f"Total examples: {len(examples)}")
print("\nBy task type:")
for task, count in Counter(e['task_type'] for e in examples).most_common():
    print(f"  {task}: {count}")
print("\n--- Sample ---")
ex = examples[0]
print(f"Task: {ex['task_type']}")
print(f"Instruction: {ex['instruction'][:200]}")
print(f"Output: {ex['output'][:300]}...")

In [ ]:
# Cell 6: Download results
from google.colab import files
files.download('/content/output/sebi_instructions.jsonl')

In [ ]:
# Cell 7: Backup to Google Drive (recommended)
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/SEBI
# !cp /content/output/sebi_instructions.jsonl /content/drive/MyDrive/SEBI/
# !cp /content/output/raw_checkpoint.jsonl /content/drive/MyDrive/SEBI/